In [3]:
import numpy as np

In [1]:
def dataset_preprocessing(path,train=True):
    seq,label,b1=[],[],[]
    d1=[30]*9 if train else [31,35,88,44,29,24,40,50,29]
    c=np.cumsum(d1)
    i=0

    for l in open(path):
        l=l.strip()
        if l=="":
            if b1:
                seq.append(np.array(b1))
                label.append(np.searchsorted(c,i))
                b1=[]
                i+=1
            else:
                b1.append(list(map(float,l.split())))
        if b1:
            seq.append(np.array(b1))
            label.append(np.searchsorted(c,i))
    return seq,np.array(label)

In [4]:
X_train,y_train=dataset_preprocessing("ae.train",True)
X_test,y_test=dataset_preprocessing("ae.test",False)

In [5]:
q=np.vstack(X_train)
mean,std=q.mean(0),q.std(0)+1e-9
X_train=[(x-mean)/std for x in X_train]
X_test=[(x-mean)/std for x in X_test]

In [6]:
def b_j(O_t,mu_j,Sigma_j):
    D=len(mu_j)
    Sigma_j = Sigma_j + 1e-3*np.eye(D)
    difference=O_t-mu_j
    inv = np.linalg.inv(Sigma_j)

    number=np.exp(-0.5*difference.T@inv@difference)
    density=np.sqrt((2*np.pi)**D*np.linalg.det(Sigma_j))

    return number/density

In [7]:
def forward(O,pi,A,mu,Sigma):
    T=len(O)
    N=len(pi)

    alpha=np.zeros((T,N))

    for i in range(1,T):
        for j in range(N):
            alpha[0,i]=pi[i]*b_j(O[0],mu[i],Sigma[j])
    return np.log(np.sum(alpha[T-1])+1e-12)

In [8]:
def backward(O,A,mu,Sigma):
    T=len(O)
    N=A.shape[0]
    beta = np.zeros((T,N))
    beta[-1]=1.0

    for t in reversed(range(T-1)):
        for i in range(N):
            beta[t,i]=np.sum(A[i,:]*[b_j(O[t+1],mu[j],Sigma[j])*beta[t+1,j] for j in range(N)])

In [9]:
class HMM:
    def __int__(self,N,D):
        self.N=N
        self.D=D
        self.pi = np.ones(N)/N
        self.A= np.ones((N,N))/N
        self.mu=np.random.randn(N.D)